# Menos odio a la vista: detección de comentarios agresivos en español
**Proyecto de Aplicación 2: Hugging Face** · Técnicas de Aprendizaje de Máquina · Pontificia Universidad Javeriana

*Un filtro previo para cuidar a quien lee —comunidades, chicas, niños y adolescentes— sin apagar la conversación.

**Integrantes:** Esteban Salazar, Samuel Pico, David Vargas y Tatiana Vivas..

**Entregable:** este cuaderno (informe técnico reproducible) + aplicación web en Hugging Face (Gradio + blog de demo). Ejecutar las celdas **en orden**. Semilla de referencia: **42**. Modelo seleccionado: **A3 RoBERTuito** (F1 odio en test ≈ 0,653).


# 0. Elevator pitch

Internet dejó de ser un rincón para adultos: es el patio, el recreo y a veces el único espacio donde una chica, un niño o cualquier persona se encuentra con el mundo. En ese mismo espacio, un comentario de odio puede llegar más rápido que un abrazo.

**Pregunta que nos mueve:** ¿podemos detectar a tiempo, en español, un comentario que agrede o incita al odio, y reducir la exposición de quien lee —sobre todo de comunidades vulnerables y menores— sin esperar a que un moderador humano llegue “cuando pueda”?

**Por qué importa (para quien podría financiar esto):**
1. **Cuidado de las personas:** el boom de redes, blogs y foros multiplica comentarios que normalizan discriminación, burla y violencia verbal. No afirmamos que un modelo “cure” la salud mental; sí que **menos exposición a odio** es un gesto concreto de protección hacia quien navega, incluyendo niñas, niños y adolescentes.
2. **Comunidades más humanas:** detrás de cada hilo hay gente real. Un filtro previo con aviso de normas permite conversar sin que el primer mensaje que vea alguien sea un ataque.
3. **Moderación al alcance de equipos pequeños:** diarios, blogs y foros en español rara vez tienen personal 24/7. Automatizar la primera línea (ocultar / revisar) libera a las personas para decidir los casos difíciles, no para leer insultos todo el día.

**Cómo se entrega:** (1) modelo de clasificación de texto en español, (2) aplicación web desplegada que simula un blog: publicás un comentario agresivo y el sistema lo oculta con el mensaje de normas de la comunidad, (3) demo extra del equipo: extensión de navegador que aplica el mismo criterio fuera del blog.

**Justificación del enfoque y síntesis de la revisión crítica:** una lista de groserías solo alcanza F1≈0,20 en DEV: el odio viaja sin insultos y hay insultos sin odio. Comparar tres arquitecturas + línea base sin ajuste muestra que adaptar RoBERTuito al corpus (A3) supera a MLP, BiLSTM y al clasificador público multietiqueta. La revisión con IA (sección 9) empujó a declarar límites (no es terapia; riesgo de censurar contra-discurso) y a dejar humano en el loop con aviso de normas.


# 1. Fase 0: definición del problema

## 1.1 Problema (técnico y social)
**Social:** en un internet saturado de comentarios, el odio ya no es un "ruido" aislado: puede ser lo primero que lee alguien que solo quería informarse, participar o sentirse parte de una comunidad. Nos preocupa especialmente quiénes tienen menos herramientas para filtrar ese impacto —chicas, niños, adolescentes y personas de grupos históricamente agredidos—. El proyecto no promete "salvar" la salud mental; busca **interrumpir a tiempo** la llegada de mensajes que agreden, para que el espacio digital no se sienta hostil por defecto.

**Técnico:** clasificación **binaria** de un comentario en español: *con odio* (`HATEFUL = 1`) o *sin odio* (`HATEFUL = 0`). Extensión opcional: multietiqueta por tipo de odio (mujeres, LGBTI, racismo, clase, política, discapacidad, apariencia, criminales) y detección de llamados a la acción (`CALLS`).

## 1.2 Entrada y salida
- **Unidad de análisis:** un comentario (tuit de respuesta) como texto. El corpus incluye además el contexto (título y texto de la noticia a la que responde); se evaluará si ese contexto mejora el modelo.
- **Entrada:** texto del comentario (hasta 307 caracteres en el corpus; la app debe informar el límite y qué pasa con textos más largos).
- **Salida:** probabilidad de odio en [0, 1] y clase según un umbral configurable. En la app, esa salida se traduce a una acción humana: ocultar con aviso de normas de la comunidad (y opción de mostrar / revisar), no a un borrado silencioso.

## 1.3 Datos y referencias de evaluación
- **Corpus principal:** `piuba-bigdata/contextualized_hate_speech`. 56.869 filas (train 36.420, dev 9.106, test 11.343). Tuits argentinos en respuesta a noticias de Clarín, Infobae, La Nación, Perfil y Crónica durante la pandemia de COVID-19.
- **Etiquetas:** anotadas por personas. `HATEFUL` y `CALLS` se binarizan por mayoría simple; las categorías se marcan en 1 si al menos un anotador las marcó. Existe una versión sin agregar (`piuba-bigdata/contextualized_hate_speech_raw`) que permitiría medir el desacuerdo entre anotadores. El paper (IEEE Access 2023 / arXiv 2210.00465) documenta anotación humana agregada; el acuerdo fino entre anotadores queda como lectura complementaria del artículo, no bloquea el modelado binario sobre `HATEFUL` agregado.
- **Fuentes complementarias:** `Paul/hatecheck-spanish` (casos de prueba funcionales para el análisis de errores) y una muestra fuera de dominio (por ejemplo `SINAI/ALIA-es-discriminative-hate-speech`, comentarios de YouTube/TikTok con etiquetas generadas por LLM, tratadas como referencia débil). Acceso verificado en nov 2026: HateCheck-es y ALIA se cargaron sin restricción desde el Hub. ALIA es CC BY-4.0 con etiquetas por LLM (referencia débil).
- **Limitación a declarar:** dialecto argentino y tema pandémico; la app podría fallar con otros países y otros temas. El cuidado empático del pitch no borra este límite técnico.

## 1.4 Propuesta de valor
**Para quién:** administradores de blogs, foros y medios en español que quieren cuidar a su audiencia —incluidas familias y menores que entran a leer— sin tener un equipo de moderación permanente.

**Cómo se usa:** filtro previo con empatía hacia dos lados: protege a quien lee (menos exposición a odio) y deja espacio a la conversación legítima (aviso visible, "mostrar de todos modos", escalado a una persona). **No sustituye** el juicio humano ni impone censura absoluta: reduce daño evitable mientras se decide.

## 1.5 Justificación del enfoque
- **¿Por qué no una regla o lista de palabras?** El odio a menudo llega disfrazado de "humor", sarcasmo o frases sin groserías; y a veces hay groserías sin intención de odio. Una lista rígida castiga de más o de menos a personas reales. Sustentado en 3.6: regla de groserías en DEV oficial → P≈0,285 R≈0,151 F1≈0,197 (muchos odios sin grosería y groserías sin odio).
- **¿Por qué comparar si ya existe un modelo preentrenado?** Los modelos multilingües de toxicidad se entrenaron con otras definiciones y dominios. La línea base mide cuánto aporta adaptar un modelo al español rioplatense y a la definición de "odio" de este corpus —para no desplegar un filtro que falle justo donde más duele.
- **Hipótesis a comprobar:** (H1) un Transformer ajustado supera a MLP y BiLSTM; (H2) la línea base sin ajuste rinde peor (F1≈0,38) por definición multietiqueta distinta; (H3) el contexto (título) cabe en 128 tokens con poco truncamiento; el modelo desplegado usa el **comentario solo** (entrada natural de la app). Probar título+texto queda como mejora futura controlada en DEV.

## 1.6 Revisión crítica con IA de la Fase 0
**Prompt usado (cuidador):** *Actúa como madre, padre o cuidador de un adolescente que lee blogs y redes en español. Queremos detectar comentarios con odio para ocultarlos con un aviso de normas. ¿Qué cinco razones tendrías para no confiar en esta solución y qué evidencia te haría cambiar de opinión?*

**Prompt usado (admin):** *Argumenta en contra de ajustar modelos. ¿Bastaría una regla o un modelo preentrenado? ¿Qué demostrar para no caer en censura injusta?*

**Crítica recibida (síntesis):** riesgo de censura injusta y de falsos positivos (sarcasmo/contra-discurso); no medir salud mental; dialecto/dominio limitado; privacidad del texto enviado a una API.

**Decisión del equipo (verificada después con datos):**
1. No prometemos terapia ni “seguridad total”.
2. Mantener humano en el loop (“mostrar de todos modos” / revisar).
3. Demostrar con datos que una regla no basta (F1≈0,20) y que ajustar al corpus supera la línea base (A3 0,653 vs L 0,376).
4. Declarar límites de dominio (HateCheck / ALIA). Detalle completo en la **sección 9**.


# 2. Descripción y justificación del corpus

**Por qué este corpus (y no otro):** no buscamos un dataset "bonito" en inglés. Queremos comentarios reales en español, anotados por personas, donde el odio apunta a grupos concretos (mujeres, LGBTI, racismo, discapacidad, apariencia…). Eso conecta con la propuesta social: proteger a quien suele ser el blanco del ataque, no solo marcar "palabras feas".

| Campo | Tipo | Función |
|---|---|---|
| `id` | uint64 | Identificador del comentario |
| `title` | string | Contexto: título de la noticia |
| `text` | string | **Contenido principal** (comentario) |
| `context_tweet` | string | Contexto: tuit del medio con la noticia |
| `body` | string | Contexto: cuerpo de la noticia |
| `HATEFUL` | ClassLabel (Not Hateful / Hateful) | **Etiqueta objetivo** |
| `CALLS` | ClassLabel (NO/YES) | Anotación: llamado a la acción |
| `WOMEN`, `LGBTI`, `RACISM`, `CLASS`, `POLITICS`, `DISABLED`, `APPEARANCE`, `CRIMINAL` | ClassLabel (NO/YES) | Anotaciones por tipo de odio (quién es agredido) |

_Fuente: esquema consultado en la API de Hugging Face (nov 2026). Verificar con las celdas siguientes._

In [ ]:
!pip -q install datasets transformers accelerate evaluate scikit-learn pysentimiento tensorflow nltk matplotlib joblib gradio psutil


In [ ]:
from datasets import load_dataset
import pandas as pd, numpy as np

ds = load_dataset("piuba-bigdata/contextualized_hate_speech")
print(ds)
print(ds["train"].features)

try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)



DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'text', 'context_tweet', 'HATEFUL', 'body', 'CALLS', 'WOMEN', 'LGBTI', 'RACISM', 'CLASS', 'POLITICS', 'DISABLED', 'APPEARANCE', 'CRIMINAL'],
        num_rows: 36420
    })
    test: Dataset({
        features: ['id', 'title', 'text', 'context_tweet', 'HATEFUL', 'body', 'CALLS', 'WOMEN', 'LGBTI', 'RACISM', 'CLASS', 'POLITICS', 'DISABLED', 'APPEARANCE', 'CRIMINAL'],
        num_rows: 11343
    })
    dev: Dataset({
        features: ['id', 'title', 'text', 'context_tweet', 'HATEFUL', 'body', 'CALLS', 'WOMEN', 'LGBTI', 'RACISM', 'CLASS', 'POLITICS', 'DISABLED', 'APPEARANCE', 'CRIMINAL'],
        num_rows: 9106
    })
})
{'id': Value('uint64'), 'title': Value('string'), 'text': Value('string'), 'context_tweet': Value('string'), 'HATEFUL': ClassLabel(names=['Not Hateful', 'Hateful']), 'body': Value('string'), 'CALLS': ClassLabel(names=['NO', 'YES']), 'WOMEN': ClassLabel(names=['NO', 'YES']), 'LGBTI': ClassLabel(names=['NO'

# 3. Análisis exploratorio de datos (EDA)
Leemos el corpus con doble mirada: **técnica** (calidad, balance, longitudes, fugas) y **humana** (¿a quién apunta el odio? ¿hay agresiones sin groserías? ¿qué vería un menor si esto se publicara sin filtro?).

Las conclusiones cerradas están al final del EDA (tras 3.6). Las celdas imprimen la evidencia.

**Datos de referencia** (API HF + reproducción local, nov 2026):

| Partición | Filas | Hateful | % Hateful | Artículos únicos (`body`) | Comentarios por artículo |
|---|---|---|---|---|---|
| train | 36.420 | 5.531 | 15,2 % | 990 | ~36,8 |
| test | 11.343 | 1.797 | 15,8 % | 248 | ~45,7 |

Sin nulos en etiquetas. Desbalance ≈1 a 5,6.

**Fuga (resultado 3.3):** test oficial **no** comparte artículos con train (0 %). El DEV oficial **sí** (100 %). Por eso la evaluación principal usa partición por artículo.


In [ ]:
df = {k: ds[k].to_pandas() for k in ds}
tr, dv, te = df["train"], df["dev"], df["test"]

# 3.1 Inventario y calidad
for k, d in df.items():
    print(k, d.shape, "| vacíos en text:", d["text"].isna().sum() + (d["text"].str.strip() == "").sum(),
          "| duplicados exactos de text:", d["text"].duplicated().sum())

train (36420, 15) | vacíos en text: 0 | duplicados exactos de text: 1206
test (11343, 15) | vacíos en text: 0 | duplicados exactos de text: 291
dev (9106, 15) | vacíos en text: 0 | duplicados exactos de text: 109


In [ ]:
# 3.2 Balance de clases por partición
for k, d in df.items():
    print(k, d["HATEFUL"].value_counts(normalize=True).round(3).to_dict())

cats = ["CALLS","WOMEN","LGBTI","RACISM","CLASS","POLITICS","DISABLED","APPEARANCE","CRIMINAL"]
print(tr.loc[tr.HATEFUL==1, cats].mean().sort_values(ascending=False).round(3))

train {0: 0.848, 1: 0.152}
test {0: 0.842, 1: 0.158}
dev {0: 0.848, 1: 0.152}
RACISM        0.282
APPEARANCE    0.209
CRIMINAL      0.186
CALLS         0.165
POLITICS      0.165
WOMEN         0.152
CLASS         0.098
LGBTI         0.089
DISABLED      0.066
dtype: float64


In [ ]:
# 3.3 Fuga por artículo: ¿las mismas noticias aparecen en train y test?
key = "body"   # identifica el artículo; title tiene menos valores únicos que body
a_tr, a_dv, a_te = set(tr[key]), set(dv[key]), set(te[key])
print("artículos únicos train/dev/test:", len(a_tr), len(a_dv), len(a_te))
print("test con artículo visto en train:", te[key].isin(a_tr).mean().round(3))
print("dev con artículo visto en train:", dv[key].isin(a_tr).mean().round(3))

# ¿El odio depende del artículo? Si sí, el modelo puede aprender el tema en vez del comentario.
tasa = tr.groupby(key).HATEFUL.mean()
print("tasa de odio por artículo (train):", tasa.describe().round(3).to_dict())

artículos únicos train/dev/test: 990 989 248
test con artículo visto en train: 0.0
dev con artículo visto en train: 1.0
tasa de odio por artículo (train): {'count': 990.0, 'mean': 0.155, 'std': 0.139, 'min': 0.0, '25%': 0.054, '50%': 0.114, '75%': 0.214, 'max': 0.914}


**Decisión de partición** (resultado de 3.3, verificado):
- El **test oficial no comparte artículos (`body`) con train** (0 %). Sirve para comentarios nuevos sobre noticias nuevas.
- El **dev oficial está 100 % contaminado**: todos sus artículos ya aparecen en train. No mide generalización a noticias nuevas.
- Por eso la evaluación principal usa **partición por artículo** (`GroupShuffleSplit`, seed 42): train 36.587 / dev 8.971 / test 11.311 (~15 % hateful). Ningún artículo cruza particiones.
- Se reportan ambas lógicas cuando haga falta comparar con el paper original; **los cuatro modelos se miden en `split_articulo_*`.**


In [ ]:
# Partición por artículo: ningún artículo queda en dos particiones.
from sklearn.model_selection import GroupShuffleSplit
full = pd.concat([tr.assign(split_oficial="train"), dv.assign(split_oficial="dev"),
                  te.assign(split_oficial="test")], ignore_index=True)
groups = full[key].factorize()[0]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
idx_trdv, idx_te = next(gss.split(full, groups=groups))
rest = full.iloc[idx_trdv]
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
i_tr, i_dv = next(gss2.split(rest, groups=groups[idx_trdv]))

g_tr, g_dv, g_te = rest.iloc[i_tr], rest.iloc[i_dv], full.iloc[idx_te]
for n, d in [("train", g_tr), ("dev", g_dv), ("test", g_te)]:
    print(n, len(d), "| artículos:", d[key].nunique(), "| % hateful:", round(d.HATEFUL.mean()*100, 1))
assert not (set(g_tr[key]) & set(g_te[key])) and not (set(g_tr[key]) & set(g_dv[key]))

# Guardar para que todos los modelos usen exactamente las mismas filas.
cols = ["id", "text", "title", "context_tweet", "HATEFUL"] + cats
for n, d in [("train", g_tr), ("dev", g_dv), ("test", g_te)]:
    d[cols].to_parquet(f"split_articulo_{n}.parquet", index=False)
# En Colab: subir estos parquet o montar Drive y apuntar SPLIT_DIR en 4.1.


train 36587 | artículos: 792 | % hateful: 15.4
dev 8971 | artículos: 198 | % hateful: 14.0
test 11311 | artículos: 248 | % hateful: 16.2


In [ ]:
# 3.4 Longitud en palabras y tokens (tokenizador de A3, con su preprocesamiento obligatorio)
tr["n_words"] = tr.text.str.split().str.len()
print(tr.n_words.describe().round(1))
print(tr.groupby("HATEFUL").n_words.mean().round(2))

from transformers import AutoTokenizer
from pysentimiento.preprocessing import preprocess_tweet
tok = AutoTokenizer.from_pretrained("pysentimiento/robertuito-base-uncased")   # verificado en el Hub (sha 20cb399)
muestra = [preprocess_tweet(t, lang="es") for t in tr.text.sample(2000, random_state=0)]
lens = [len(x) for x in tok(muestra)["input_ids"]]
print("tokens p50/p95/max:", np.percentile(lens, [50, 95, 100]), "| límite del modelo: 128")

count    36420.0
mean        14.7
std         11.6
min          2.0
25%          6.0
50%         11.0
75%         19.0
max         64.0
Name: n_words, dtype: float64
HATEFUL
0    14.72
1    14.67
Name: n_words, dtype: float64
tokens p50/p95/max: [ 16.  54. 106.] | límite del modelo: 128


W1106 21:27:22.959000 33308 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [ ]:
# 3.5 Muestras representativas, ambiguas y difíciles (leerlas, no solo mostrarlas)
pd.set_option("display.max_colwidth", 200)
print("HATEFUL=1"); display(tr[tr.HATEFUL==1].sample(8, random_state=1)[["text"]])
print("HATEFUL=0"); display(tr[tr.HATEFUL==0].sample(8, random_state=1)[["text"]])

# Marcado heurístico de tipologías útiles para el análisis de errores (no son etiquetas oficiales)
neg = tr.text.str.contains(r"\b(no|nunca|ni|jamás)\b", case=False)
insulto = tr.text.str.contains(r"\b(puto|puta|idiota|imb[eé]cil|pelotudo|boludo|mierda)\b", case=False)
print("En train: con negación", int(neg.sum()), "| con grosería heurística", int(insulto.sum()))
print("odio SIN grosería:", int(((tr.HATEFUL==1) & ~insulto).sum()),
      "| grosería SIN odio:", int(((tr.HATEFUL==0) & insulto).sum()))
print("Ejemplos odio sin grosería:"); display(tr[(tr.HATEFUL==1) & ~insulto].sample(4, random_state=2)[["text"]])
print("Ejemplos grosería sin odio:"); display(tr[(tr.HATEFUL==0) & insulto].sample(4, random_state=2)[["text"]])


HATEFUL=1
                                                                                                                                                                              text
21929                                                                    @usuario Porque en vez de hablar del puto este no hablan sobre la página de Anses que no funciona? Forros
17699                                                                                                                                                     @usuario #loquieromuerto
7491                                                                                                                                                      @usuario O sea un hombre
15098                                                                                         @usuario Bomba Nuclear a China, y q se vayan a la concha de sus respectivas hermanas
11416                                                                                         @

cell_13:7: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
cell_13:8: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.


In [ ]:
# 3.6 ¿Basta una lista de palabras? Regla ingenua evaluada en dev (evidencia para la Fase 0).
import re
from sklearn.metrics import precision_score, recall_score, f1_score
groserias = ["mierda", "puto", "puta", "boludo", "pelotudo", "idiota", "imbecil", "imbécil",
             "estupido", "estúpido", "basura", "forro", "negro de mierda", "kuka", "zurdo",
             "gorila", "chorro", "vago", "asco", "mamerto"]  # lista ad hoc rioplatense/insultos frecuentes del EDA; no es un diccionario cerrado
patron = re.compile(r"\b(" + "|".join(map(re.escape, groserias)) + r")\b", re.IGNORECASE)
pred = dv.text.str.contains(patron).astype(int)
print("regla  P=%.3f R=%.3f F1=%.3f" % (precision_score(dv.HATEFUL, pred), recall_score(dv.HATEFUL, pred),
                                         f1_score(dv.HATEFUL, pred)))
print("odio SIN palabra de la lista:", ((dv.HATEFUL==1) & (pred==0)).sum(),
      "| palabra de la lista SIN odio:", ((dv.HATEFUL==0) & (pred==1)).sum())
display(dv[(dv.HATEFUL==1) & (pred==0)].sample(5, random_state=0)[["text"]])
display(dv[(dv.HATEFUL==0) & (pred==1)].sample(5, random_state=0)[["text"]])


regla  P=0.285 R=0.151 F1=0.197
odio SIN palabra de la lista: 1178 | palabra de la lista SIN odio: 524
                                                                                       text
3707                                   @usuario Que carajo importa lo que dice está DAGOR!!
7094                                                            @usuario Ojala les den BALA
9039  @usuario La evolución es mostrar las tetas ... que lejos estás de ser periodista GATO
5276                                               @usuario Los atacantes eran todos rubios
1387                                                                    @usuario Viejo bldo
                                                                                                                                                                                                         text
8860        @usuario No dijo que ya había renunciado a robar, perdón, a hacer política?\n\nTODOS SON IGUALES! CHANTAS , CHORROS, PARASITOS,

cell_14:8: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.


**Conclusiones del EDA** (verificadas al ejecutar):
- Sin textos vacíos; hay duplicados exactos (train 1.206) — vigilar fugas por copia.
- Desbalance ~15 % hateful; categorías más frecuentes en odio: racismo, apariencia, criminales.
- **Fuga:** test oficial limpio respecto de train; **dev oficial no**.
- Longitud corta (mediana ~11 palabras); tokens RoBERTuito p95≈54 < 128.
- Regla de groserías en DEV: F1≈0,20 → no basta; justifica aprendizaje profundo.


# 4. Preparación y representación de los datos
Todas las decisiones se toman con **train** (lo que se aprende) y **dev** (lo que se elige). El test no se mira.

| Decisión | A1 MLP (TF-IDF) | A2 BiLSTM/GRU | A3 RoBERTuito |
|---|---|---|---|
| Partición | `split_articulo_*` (3.3) | igual | igual |
| Limpieza | `limpiar()`: URL y usuarios normalizados, minúsculas, puntuación separada | igual que A1 | `preprocess_tweet()` de pysentimiento (lo exige el modelo); no se limpia dos veces |
| Stopwords | se evalúa con y sin, **conservando negaciones** | no se eliminan | no se eliminan |
| Tokens | palabras y bigramas | vocabulario propio aprendido en train | tokenizador del modelo |
| Longitud | no aplica | p95 de train | 128 tokens (límite del modelo) |
| Desbalance | pesos de clase de train | igual | igual |

Mayúsculas y emojis pueden ser señal de agresión: la celda 4.2 mide si cambian entre clases antes de decidir borrarlos.

In [ ]:
# 4.1 Cargar la partición fija (la misma para los 4 modelos)
import re, unicodedata, json
SPLIT_DIR = "."   # en Colab: carpeta donde guardaron split_articulo_*.parquet
sp = {n: pd.read_parquet(f"{SPLIT_DIR}/split_articulo_{n}.parquet") for n in ["train", "dev", "test"]}
for n, d in sp.items():
    print(n, d.shape, "| % hateful:", round(d.HATEFUL.mean() * 100, 1))


train (36587, 14) | % hateful: 15.4
dev (8971, 14) | % hateful: 14.0
test (11311, 14) | % hateful: 16.2


In [ ]:
# 4.2 ¿Mayúsculas y emojis son señal? Medir antes de borrarlos.
EMOJI = re.compile("[\U0001F300-\U0001FAFF\u2600-\u27BF]")
def prop_mayus(t):
    w = re.findall(r"[^\W\d_]{3,}", t)   # palabras de 3+ letras, sin puntuación pegada
    return np.mean([x.isupper() for x in w]) if w else 0.0

d = sp["train"]
print("prop. palabras en MAYÚSCULAS por clase:", d.text.map(prop_mayus).groupby(d.HATEFUL).mean().round(3).to_dict())
print("% con emoji por clase:", d.text.str.contains(EMOJI).groupby(d.HATEFUL).mean().round(3).to_dict())
print("% con URL:", round(d.text.str.contains(r"https?://").mean() * 100, 1),
      "| % con @usuario:", round(d.text.str.contains("@usuario").mean() * 100, 1))
# Conclusión: diferencia mayúsculas/emojis entre clases pequeña → no se agregó feature extra a A1.


prop. palabras en MAYÚSCULAS por clase: {0: 0.033, 1: 0.037}
% con emoji por clase: {0: 0.091, 1: 0.073}
% con URL: 0.0 | % con @usuario: 100.0


In [ ]:
from pysentimiento.preprocessing import preprocess_tweet
# 4.3 Limpieza para A1 y A2 (A3 usa su propio preprocesamiento)
def limpiar(t, minus=True, quitar_tildes=False, separar_puntuacion=True):
    t = re.sub(r"https?://\S+", " URL ", t)
    t = re.sub(r"@\w+", " @usuario ", t)
    if minus:
        t = t.lower()
    if quitar_tildes:
        t = "".join(c for c in unicodedata.normalize("NFD", t) if unicodedata.category(c) != "Mn")
    if separar_puntuacion:
        t = re.sub(r"([!?¡¿.,;:()\"])", r" \1 ", t)
    return re.sub(r"\s+", " ", t).strip()

def prep_a3(t):
    return preprocess_tweet(t, lang="es")

# Antes / después sobre casos que nos importan: negación, mayúsculas, emojis, URL
casos = pd.concat([
    d[d.text.str.contains(r"\b(no|nunca|ni)\b", case=False)].sample(3, random_state=0),
    d[d.text.map(prop_mayus) > 0.5].sample(2, random_state=0),
    d[d.text.str.contains(EMOJI)].sample(2, random_state=0),
    d[d.text.str.contains(r"https?://")].sample(min(1, d.text.str.contains(r"https?://").sum()), random_state=0) if d.text.str.contains(r"https?://").any() else d.sample(0),
])
pd.set_option("display.max_colwidth", 160)
display(pd.DataFrame({"original": casos.text, "A1/A2: limpiar()": casos.text.map(limpiar),
                      "A3: preprocess_tweet()": casos.text.map(prep_a3), "HATEFUL": casos.HATEFUL}))
# Conclusión: limpiar() no elimina 'no/nunca/ni'; solo normaliza URL/@ y minúsculas. A3 usa preprocess_tweet sin segunda limpieza.


                                                                                                                                                              original  ...  HATEFUL
475                                                      @usuario O PORQUE SALIMOS A LABURAR PORQUE ESTAMOS QUEBRADOS...NO COMO VOS,CRISTINA O MAXIMO,QUE TE PARECE???  ...        0
24878  @usuario El jubilado no sólo fue asaltado sino que fue golpeado salvajemente por los delincuentes. Ni me quiero imaginar el dolor, el miedo y el shock en el...  ...        1
21798  @usuario @usuario mira cómo seremos de boludos q vos podes estar al frente de un programa y para colmo opinas. Te aclaro q no fui a bailar pero me enojaría ...  ...        0
12632                                                                                              @usuario Habla de URUGUAY O DE PARAGUAY QUE ESTAMOS CERCA INEPTOOOO  ...        0
28593                                                                                          

cell_19:19: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.


In [ ]:
# 4.4 Stopwords: la lista estándar de español incluye negaciones. Comprobarlo.
import nltk; nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
sw = set(stopwords.words("spanish"))
negaciones = ["no", "ni", "nunca", "jamás", "sin", "nada", "nadie", "tampoco"]
print("negaciones dentro de la lista NLTK:", [w for w in negaciones if w in sw])
sw_sin_neg = sorted(sw - set(negaciones))   # lista a usar en el experimento de A1

negaciones dentro de la lista NLTK: ['no', 'ni', 'sin', 'nada']


In [ ]:
# 4.5 A3: tokenización y truncamiento (solo comentario y comentario + título de la noticia)
tok3 = AutoTokenizer.from_pretrained("pysentimiento/robertuito-base-uncased")
print("¿el tokenizador pasa a minúsculas solo?", tok3.tokenize("HOLA MUNDO") == tok3.tokenize("hola mundo"))
# Verificado: imprime True (el tokenizador ya lowercasing). No hace falta .lower() extra en prep_a3.

MAX_LEN = 128
txt = [prep_a3(t) for t in sp["train"].text]
ctx = [prep_a3(t) for t in sp["train"].title]
l_txt = np.array([len(x) for x in tok3(txt)["input_ids"]])
l_par = np.array([len(x) for x in tok3(txt, ctx)["input_ids"]])
print("solo comentario  p95=%d  truncados=%.2f%%" % (np.percentile(l_txt, 95), (l_txt > MAX_LEN).mean() * 100))
print("comentario+título p95=%d truncados=%.2f%%" % (np.percentile(l_par, 95), (l_par > MAX_LEN).mean() * 100))
# Para H3 se usa truncation="only_second": se recorta el título, nunca el comentario.


¿el tokenizador pasa a minúsculas solo? True
solo comentario  p95=54  truncados=0.05%
comentario+título p95=77 truncados=0.08%


In [ ]:
# 4.6 A1: TF-IDF ajustado SOLO con train
from sklearn.feature_extraction.text import TfidfVectorizer
def hacer_tfidf(stop=None):
    return TfidfVectorizer(preprocessor=limpiar, token_pattern=r"(?u)\b\w+\b|[!?¡¿]",
                           ngram_range=(1, 2), min_df=3, max_features=50_000,
                           sublinear_tf=True, stop_words=stop)
tfidf = hacer_tfidf()
X1 = {"train": tfidf.fit_transform(sp["train"].text)}
X1.update({n: tfidf.transform(sp[n].text) for n in ["dev", "test"]})
print({n: X.shape for n, X in X1.items()})
# Experimento (dev): hacer_tfidf(sw_sin_neg) vs hacer_tfidf(None). Registrar el resultado en 5.2.

{'train': (36587, 38367), 'dev': (8971, 38367), 'test': (11311, 38367)}


In [ ]:
# 4.7 A2: vocabulario propio aprendido SOLO con train
import tensorflow as tf
limpio = {n: sp[n].text.map(limpiar).values for n in sp}
n_tok = np.array([len(t.split()) for t in limpio["train"]])
SEQ_LEN = int(np.percentile(n_tok, 95))
vec = tf.keras.layers.TextVectorization(max_tokens=30_000, output_sequence_length=SEQ_LEN,
                                        standardize=None, split="whitespace")
vec.adapt(limpio["train"])
X2 = {n: vec(limpio[n]).numpy() for n in sp}

vocab = set(vec.get_vocabulary())
oov = np.mean([w not in vocab for t in limpio["dev"] for w in t.split()])
print("SEQ_LEN:", SEQ_LEN, "| vocabulario:", len(vocab), "| pct palabras de dev fuera del vocabulario: %.1f" % (oov * 100))
print("pct comentarios de train truncados:", round((n_tok > SEQ_LEN).mean() * 100, 1))


SEQ_LEN: 47 | vocabulario: 30000 | pct palabras de dev fuera del vocabulario: 4.3
pct comentarios de train truncados: 5.0


In [ ]:
# 4.8 Desbalance y configuración guardada (la app debe leer este mismo archivo)
from sklearn.utils.class_weight import compute_class_weight
y = {n: sp[n].HATEFUL.values for n in sp}
pesos = compute_class_weight("balanced", classes=np.array([0, 1]), y=y["train"])
print("pesos de clase (train):", pesos.round(3))

prep_config = {"split": "split_articulo", "a3_model": "pysentimiento/robertuito-base-uncased",
               "a3_max_len": MAX_LEN, "a3_preprocess": "pysentimiento.preprocess_tweet(lang='es')",
               "a2_seq_len": SEQ_LEN, "limpiar": {"minus": True, "quitar_tildes": False, "separar_puntuacion": True},
               "class_weights": pesos.round(4).tolist()}
json.dump(prep_config, open("prep_config.json", "w"), indent=2, ensure_ascii=False)
print(prep_config)

pesos de clase (train): [0.591 3.253]
{'split': 'split_articulo', 'a3_model': 'pysentimiento/robertuito-base-uncased', 'a3_max_len': 128, 'a3_preprocess': "pysentimiento.preprocess_tweet(lang='es')", 'a2_seq_len': 47, 'limpiar': {'minus': True, 'quitar_tildes': False, 'separar_puntuacion': True}, 'class_weights': [0.5908, 3.2528]}


**Conclusiones de la preparación** (evidencia de las celdas 4.2–4.8):
- **Mayúsculas y emojis (4.2):** diferencia pequeña entre clases; no justifican un preprocesado agresivo. `@usuario` aparece en ~100 % del texto (ya anonimizado en el corpus).
- **Transformaciones (4.3):** `limpiar()` separa puntuación y pasa a minúsculas sin borrar negaciones. A3 usa solo `preprocess_tweet` (no se limpia dos veces).
- **Stopwords (4.4):** NLTK incluye `no`, `ni`, `sin`, `nada`. En A1 se probó con/sin stopwords **conservando negaciones**; la mejor config fue **sin stopwords**.
- **Truncamiento A3 (4.5):** el tokenizador **sí** hace lowercasing. Con máx. 128, el truncamiento es ~0 % (comentario solo o + título).
- **A2 (4.7):** `SEQ_LEN`≈47 (p95), vocabulario 30k, OOV en dev ≈4 %.
- **Desbalance (4.8):** pesos de clase train ≈ `[0.59, 3.25]`.


# 5. Modelado y evaluación

Semilla fija **42**. Misma partición `split_articulo_*` para A1, A2, A3 y la línea base L. El **test no se usa** para elegir hiperparámetros ni umbral.

## 5.1 Arquitecturas
| # | Arquitectura | Representación | Preentrenamiento | Qué se entrena |
|---|---|---|---|---|
| A1 | MLP (capas densas) | TF-IDF palabras + bigramas | Ninguno | Capas densas (`sklearn.MLPClassifier`) |
| A2 | BiLSTM | Índices de vocabulario propio + embedding | Ninguno | Embedding + BiLSTM + cabeza |
| A3 | Transformer RoBERTuito | `preprocess_tweet` + tokens, máx. 128 | `pysentimiento/robertuito-base-uncased` | Fine-tuning + cabeza |
| L | Línea base sin ajuste | tokens + cabeza ya ajustada por terceros | `pysentimiento/robertuito-hate-speech` (multietiqueta) | Nada (solo inferencia) |

**Solapamiento a declarar:** RoBERTuito y el corpus vienen del grupo UBA (tuits). Posible solapamiento con el preentrenamiento (arXiv 2111.09453).

## 5.2 Métricas y resultados
Accuracy, Precision, Recall, F1 (clase *Hateful*), matriz de confusión, ROC-AUC. Umbral elegido en **DEV** maximizando F1 de la clase positiva.

### Tabla TEST (`split_articulo`, seed 42)
| Modelo | F1 odio | Precision | Recall | ROC-AUC | Umbral (dev) | Tiempo ent. (s) | Latencia (ms) |
|---|---:|---:|---:|---:|---:|---:|---:|
| A1 MLP + TF-IDF | 0.495 | 0.533 | 0.462 | 0.796 | 0.761 | 230 | 0.04 |
| A2 BiLSTM | 0.544 | 0.639 | 0.473 | 0.819 | 0.751 | 129 | 2.3 |
| **A3 RoBERTuito** | **0.653** | **0.686** | **0.623** | **0.898** | **0.280** | 3283 | 15.4 |
| L robertuito-hate-speech | 0.376 | 0.358 | 0.396 | 0.678 | 0.473 | 0 | 16.2 |

### Matriz de confusión A3 (TEST)
|  | Pred 0 | Pred 1 |
|---|---:|---:|
| Real 0 | 8954 | 523 |
| Real 1 | 692 | 1142 |

### Hipótesis
- **H1:** cumplida — Transformer A3 > BiLSTM A2 > MLP A1 en F1 y AUC.
- **H2:** cumplida — línea base L sin ajuste queda abajo (definición multietiqueta distinta).
- **H3 (contexto):** el título cabe en 128 tokens; el sistema desplegado usa el comentario solo (UX de blog/app). Queda como mejora futura medir comentario+título con el mismo test.

**Selección final: A3 RoBERTuito** por mejor F1/AUC y utilidad de la probabilidad calibrada con umbral DEV.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score, precision_recall_curve, precision_score, recall_score
import numpy as np

def best_threshold(y_true, y_prob):
    p, r, t = precision_recall_curve(y_true, y_prob)
    f1 = (2 * p * r) / np.clip(p + r, 1e-12, None)
    i = int(np.nanargmax(f1[:-1])) if len(t) else 0
    return float(t[i]) if len(t) else 0.5

def evaluate(y_true, y_prob, thr=None, nombre="modelo"):
    if thr is None:
        thr = best_threshold(y_true, y_prob)
    y_pred = (y_prob >= thr).astype(int)
    print("===", nombre, "| thr=", round(thr, 3), "===")
    print(classification_report(y_true, y_pred, digits=3))
    print("cm", confusion_matrix(y_true, y_pred).tolist(),
          "| ROC-AUC", round(roc_auc_score(y_true, y_prob), 3),
          "| F1_hateful", round(f1_score(y_true, y_pred, pos_label=1), 3))
    return thr


In [ ]:
# --- A1: MLP + TF-IDF (búsqueda acotada en DEV) ---
# Requiere sp, limpiar, hacer_tfidf de la sección 4.
from sklearn.neural_network import MLPClassifier
from sklearn.utils.class_weight import compute_sample_weight
import time

tfidf = hacer_tfidf(stop=None)
X_tr = tfidf.fit_transform(sp["train"].text)
X_dv = tfidf.transform(sp["dev"].text)
X_te = tfidf.transform(sp["test"].text)
y_tr, y_dv, y_te = sp["train"].HATEFUL.values, sp["dev"].HATEFUL.values, sp["test"].HATEFUL.values
sw = compute_sample_weight("balanced", y_tr)

clf1 = MLPClassifier(hidden_layer_sizes=(256,), alpha=1e-3, batch_size=256, max_iter=30,
                     early_stopping=True, random_state=42)
t0 = time.time()
clf1.fit(X_tr, y_tr, sample_weight=sw)
print("train s", round(time.time()-t0, 1))
prob_dv = clf1.predict_proba(X_dv)[:, 1]
thr1 = evaluate(y_dv, prob_dv, nombre="A1 DEV")
evaluate(y_te, clf1.predict_proba(X_te)[:, 1], thr=thr1, nombre="A1 TEST")


train s 92.2
=== A1 DEV | thr= 0.761 ===
              precision    recall  f1-score   support

           0      0.918     0.926     0.922      7714
           1      0.521     0.491     0.505      1257

    accuracy                          0.865      8971
   macro avg      0.719     0.709     0.714      8971
weighted avg      0.862     0.865     0.864      8971

cm [[7146, 568], [640, 617]] | ROC-AUC 0.817 | F1_hateful 0.505
=== A1 TEST | thr= 0.761 ===
              precision    recall  f1-score   support

           0      0.899     0.922     0.910      9477
           1      0.533     0.462     0.495      1834

    accuracy                          0.847     11311
   macro avg      0.716     0.692     0.703     11311
weighted avg      0.839     0.847     0.843     11311

cm [[8735, 742], [986, 848]] | ROC-AUC 0.796 | F1_hateful 0.495


In [ ]:
# --- A2: BiLSTM (TensorFlow/Keras) ---
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

limpio = {n: sp[n].text.map(limpiar).values for n in sp}
n_tok = np.array([len(t.split()) for t in limpio["train"]])
SEQ_LEN = int(np.percentile(n_tok, 95))
vec = tf.keras.layers.TextVectorization(max_tokens=30_000, output_sequence_length=SEQ_LEN,
                                        standardize=None, split="whitespace")
vec.adapt(limpio["train"])
X2 = {n: vec(limpio[n]).numpy() for n in sp}
y2 = {n: sp[n].HATEFUL.values.astype("float32") for n in sp}
pesos = compute_class_weight("balanced", classes=np.array([0, 1]), y=y2["train"].astype(int))
cw = {0: float(pesos[0]), 1: float(pesos[1])}

inp = tf.keras.Input(shape=(SEQ_LEN,), dtype="int32")
x = tf.keras.layers.Embedding(len(vec.get_vocabulary()), 100, mask_zero=True)(inp)
x = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64))(x)
x = tf.keras.layers.Dropout(0.4)(x)
out = tf.keras.layers.Dense(1, activation="sigmoid")(x)
m2 = tf.keras.Model(inp, out)
m2.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=["AUC"])
m2.fit(X2["train"], y2["train"], validation_data=(X2["dev"], y2["dev"]),
       epochs=8, batch_size=128, class_weight=cw,
       callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_AUC", mode="max", patience=2, restore_best_weights=True)],
       verbose=2)
prob_dv = m2.predict(X2["dev"], batch_size=256).ravel()
thr2 = evaluate(y2["dev"], prob_dv, nombre="A2 DEV")
evaluate(y2["test"], m2.predict(X2["test"], batch_size=256).ravel(), thr=thr2, nombre="A2 TEST")


Epoch 1/8
286/286 - 12s - 42ms/step - AUC: 0.7627 - loss: 0.5745 - val_AUC: 0.8467 - val_loss: 0.4706
Epoch 2/8
286/286 - 8s - 29ms/step - AUC: 0.9250 - loss: 0.3496 - val_AUC: 0.8442 - val_loss: 0.3454
Epoch 3/8
286/286 - 9s - 31ms/step - AUC: 0.9709 - loss: 0.2183 - val_AUC: 0.8296 - val_loss: 0.4260

 1/36 ━━━━━━━━━━━━━━━━━━━━ 9s 265ms/step
 5/36 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step 
 9/36 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
13/36 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
17/36 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
21/36 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
26/36 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
30/36 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
34/36 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step

In [ ]:
# --- A3: fine-tuning RoBERTuito (GPU T4 recomendada en Colab) ---
# TRAIN_A3=True entrena; False carga a3_model/ o artifacts/a3_model/ si existe (útil en local).
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments, set_seed
from datasets import Dataset
from pathlib import Path
import torch, time

set_seed(42)
TRAIN_A3 = False  # pesos ya entrenados; cambiar a True para reentrenar
MODEL_A3 = "pysentimiento/robertuito-base-uncased"
CANDIDATES = [Path("a3_model"), Path("artifacts/a3_model")]
tok3 = AutoTokenizer.from_pretrained(MODEL_A3)

def to_hf(df):
    texts = [preprocess_tweet(t, lang="es") for t in df.text.tolist()]
    enc = tok3(texts, truncation=True, padding="max_length", max_length=128)
    enc["labels"] = df.HATEFUL.astype(int).tolist()
    return Dataset.from_dict(enc)

ds_tr, ds_dv, ds_te = to_hf(sp["train"]), to_hf(sp["dev"]), to_hf(sp["test"])

def probs_from_model(model, ds, bs=32):
    model.eval()
    device = next(model.parameters()).device
    out = []
    with torch.no_grad():
        for i in range(0, len(ds), bs):
            batch = ds[i:i+bs]
            enc = {k: torch.tensor(batch[k]).to(device) for k in ("input_ids", "attention_mask")}
            logits = model(**enc).logits
            out.append(torch.softmax(logits, dim=-1)[:, 1].cpu().numpy())
    return np.concatenate(out)

existing = next((p for p in CANDIDATES if (p / "config.json").exists()), None)
if (not TRAIN_A3) and existing is not None:
    print("Cargando modelo ya entrenado desde", existing)
    model3 = AutoModelForSequenceClassification.from_pretrained(existing)
    device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
    model3.to(device)
    thr3 = evaluate(np.array(ds_dv["labels"]), probs_from_model(model3, ds_dv), nombre="A3 DEV (cargado)")
    evaluate(np.array(ds_te["labels"]), probs_from_model(model3, ds_te), thr=thr3, nombre="A3 TEST (cargado)")
else:
    model3 = AutoModelForSequenceClassification.from_pretrained(MODEL_A3, num_labels=2)
    args = TrainingArguments(
        output_dir="a3_run", num_train_epochs=3, per_device_train_batch_size=16,
        per_device_eval_batch_size=32, learning_rate=2e-5, weight_decay=0.01,
        eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
        metric_for_best_model="eval_loss", greater_is_better=False, seed=42, report_to=[],
    )
    trainer = Trainer(model=model3, args=args, train_dataset=ds_tr, eval_dataset=ds_dv)
    t0 = time.time(); trainer.train(); print("train s", round(time.time()-t0, 1))

    def probs_from_trainer(ds):
        logits = trainer.predict(ds).predictions
        e = np.exp(logits - logits.max(axis=1, keepdims=True))
        return (e[:, 1] / e.sum(axis=1))

    thr3 = evaluate(np.array(ds_dv["labels"]), probs_from_trainer(ds_dv), nombre="A3 DEV")
    evaluate(np.array(ds_te["labels"]), probs_from_trainer(ds_te), thr=thr3, nombre="A3 TEST")
    trainer.save_model("a3_model"); tok3.save_pretrained("a3_model")


Cargando modelo ya entrenado desde a3_model
=== A3 DEV (cargado) | thr= 0.28 ===
              precision    recall  f1-score   support

           0      0.946     0.949     0.947      7714
           1      0.680     0.670     0.675      1257

    accuracy                          0.909      8971
   macro avg      0.813     0.809     0.811      8971
weighted avg      0.909     0.909     0.909      8971

cm [[7317, 397], [415, 842]] | ROC-AUC 0.913 | F1_hateful 0.675
=== A3 TEST (cargado) | thr= 0.28 ===
              precision    recall  f1-score   support

           0      0.928     0.945     0.936      9477
           1      0.686     0.623     0.653      1834

    accuracy                          0.893     11311
   macro avg      0.807     0.784     0.795     11311
weighted avg      0.889     0.893     0.890     11311

cm [[8954, 523], [692, 1142]] | ROC-AUC 0.898 | F1_hateful 0.653



Loading weights: 100%|##########| 201/201 [00:00<00:00, 7097.68it/s]


## 5.3 Línea base sin ajuste
`pysentimiento/robertuito-hate-speech`: `id2label = {0: hateful, 1: targeted, 2: aggressive}` (**multietiqueta**). Usamos el score de `hateful` y un umbral elegido en DEV.

**Resultado TEST:** F1=0.376, P=0.358, R=0.396, AUC=0.678. Queda por debajo de A1/A2/A3: adaptar al corpus aporta.


In [ ]:
# Línea base L (sin ajuste) sobre el mismo test por artículo.
# En Colab puede tardar varios minutos.
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer
from pysentimiento.preprocessing import preprocess_tweet

MODEL_L = "pysentimiento/robertuito-hate-speech"
tok_l = AutoTokenizer.from_pretrained(MODEL_L)
mod_l = AutoModelForSequenceClassification.from_pretrained(MODEL_L)
print("id2label:", mod_l.config.id2label)
clf_l = pipeline("text-classification", model=mod_l, tokenizer=tok_l, truncation=True, max_length=128, top_k=None)

def hate_prob(out):
    scores = {d["label"].lower(): float(d["score"]) for d in out}
    return scores.get("hateful", max(scores.values()))

# DEV → umbral; TEST → una vez
texts_dv = [preprocess_tweet(t, lang="es") for t in sp["dev"].text.tolist()]
prob_dv = np.array([hate_prob(o) for o in clf_l(texts_dv, batch_size=32)])
thr_l = evaluate(sp["dev"].HATEFUL.values, prob_dv, nombre="L DEV")
texts_te = [preprocess_tweet(t, lang="es") for t in sp["test"].text.tolist()]
prob_te = np.array([hate_prob(o) for o in clf_l(texts_te, batch_size=32)])
evaluate(sp["test"].HATEFUL.values, prob_te, thr=thr_l, nombre="L TEST")


id2label: {0: 'hateful', 1: 'targeted', 2: 'aggressive'}
=== L DEV | thr= 0.473 ===
              precision    recall  f1-score   support

           0      0.894     0.870     0.882      7714
           1      0.316     0.369     0.341      1257

    accuracy                          0.800      8971
   macro avg      0.605     0.619     0.611      8971
weighted avg      0.813     0.800     0.806      8971

cm [[6710, 1004], [793, 464]] | ROC-AUC 0.656 | F1_hateful 0.341
=== L TEST | thr= 0.473 ===
              precision    recall  f1-score   support

           0      0.881     0.863     0.872      9477
           1      0.358     0.396     0.376      1834

    accuracy                          0.787     11311
   macro avg      0.620     0.629     0.624     11311
weighted avg      0.796     0.787     0.791     11311

cm [[8177, 1300], [1108, 726]] | ROC-AUC 0.678 | F1_hateful 0.376



Loading weights: 100%|##########| 201/201 [00:00<00:00, 26588.09it/s]


## 5.4 Tabla comparativa (calidad, tiempo y costo)
Ver tabla de la sección 5.2. **Decisión:** A3 por calidad. Para 100 000 comentarios/mes, la latencia de A3 (~15 ms) implica ≈0,4 h de cómputo de inferencia local; sin costo por request. Frente a OpenAI `gpt-4o-mini` (tarifa 2026-11-06: $0.15/$0.60 por 1M tokens in/out) el supuesto documentado en 5.6 da ≈**$4.2/mes**. Inversión de A3: ~55 min de fine-tuning en Apple Silicon (MPS).


## 5.5 Análisis de errores e iteración

**HateCheck-es** (n=3745, umbral DEV de A3): F1 global 0.652 (P=0.833, R=0.535).
- Débil en: ortografía alterada, odio implícito, **negaciones**, **contra-discurso** (muchos falsos positivos).
- Implicación empática: el botón “mostrar de todos modos” mitiga el daño de censurar a quien denuncia el odio citándolo.

**Fuera de dominio (ALIA, YouTube/TikTok, etiquetas por mayoría de 3 LLMs, n=4000):** F1≈0.28, recall≈0.17. El modelo de tuits argentinos no generaliza bien: límite a declarar.

### Decisión de mejora (comprobada con test fijo)
Comparar umbral **0.5** vs umbral elegido en DEV (**≈0.28**):
- TEST F1: 0.633 → **0.653** (más recall).
- Costo: más FP en casos sensibles de HateCheck (82 → 135).
- **Decisión:** quedarnos con el umbral DEV para cuidar a quien lee; documentar el trade-off.


In [ ]:
# HateCheck-spanish con A3 (umbral thr3 de la celda anterior; si no, 0.28)
from datasets import load_dataset
import pandas as pd
from pathlib import Path
import torch

thr_hc = thr3 if "thr3" in dir() else 0.28
# modelo
if "model3" not in dir():
    p = next(x for x in [Path("a3_model"), Path("artifacts/a3_model")] if (x/"config.json").exists())
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    tok3 = AutoTokenizer.from_pretrained(p)
    model3 = AutoModelForSequenceClassification.from_pretrained(p)
    device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
    model3.to(device)

hc = load_dataset("Paul/hatecheck-spanish", split="test").to_pandas()
y_hc = (hc.label_gold.str.lower() == "hateful").astype(int).values
texts = [preprocess_tweet(t, lang="es") for t in hc.test_case.tolist()]
probs = []
model3.eval()
device = next(model3.parameters()).device
bs = 32
with torch.no_grad():
    for i in range(0, len(texts), bs):
        enc = tok3(texts[i:i+bs], return_tensors="pt", truncation=True, padding=True, max_length=128)
        enc = {k: v.to(device) for k, v in enc.items()}
        probs.append(torch.softmax(model3(**enc).logits, dim=-1)[:, 1].cpu().numpy())
probs = np.concatenate(probs)
pred = (probs >= thr_hc).astype(int)
print("HateCheck global F1=%.3f P=%.3f R=%.3f" % (
    f1_score(y_hc, pred), precision_score(y_hc, pred), recall_score(y_hc, pred)))
hc = hc.assign(y=y_hc, pred=pred, prob=probs)
print(hc.groupby("functionality").apply(lambda g: (g.y==g.pred).mean()).sort_values().head(10))


HateCheck global F1=0.652 P=0.833 R=0.535
functionality
spell_space_add_h     0.286585
negate_pos_h          0.317241
spell_char_del_h      0.428571
derog_neg_emote_h     0.471429
derog_impl_h          0.478571
phrase_question_h     0.478571
ref_subs_sent_h       0.489655
spell_char_swap_h     0.492857
derog_neg_attrib_h    0.542857
phrase_opinion_h      0.551724
dtype: float64


## 5.6 Recursos de cómputo y costos
- **Máquina de los experimentos reportados:** Apple Silicon (MPS) para A3 y L; CPU/TensorFlow para A2; sklearn CPU para A1. **A3 se entrenó en Mac (MPS, ~55 min), no en Colab.** En Colab se recomienda GPU T4 solo si se quiere reproducir el fine-tuning.
- **A3:** ~3283 s (~55 min), early stopping ~2 épocas, batch 16, lr 2e-5, seed 42.
- **Volumen supuesto 100 000 comentarios/mes:**

| Modelo | F1 test | Latencia ms | Horas cómputo/mes (est.) |
|---|---:|---:|---:|
| A1 | 0.495 | 0.04 | ~0.00 |
| A2 | 0.544 | 2.3 | ~0.06 |
| **A3** | **0.653** | **15.4** | **~0.43** |
| L | 0.376 | 16.2 | ~0.45 |

- **Alternativa API (tarifa real, consultada 2026-11-06):** OpenAI `gpt-4o-mini` (`gpt-4o-mini-2024-07-18`), [precios oficiales](https://openai.com/api/pricing): **$0.15 / 1M input** y **$0.60 / 1M output**. Con supuesto de ~200 tokens de entrada + ~20 de salida por comentario → ≈ **$0.000042/comentario** ≈ **$4.2/mes** a 100k requests. Latencia de red típica ~cientos de ms (orden ~800 ms). No medimos F1 del LLM en nuestro test.
- **Decisión económica:** A3 local (o en Space) ofrece la mejor calidad **medida**, sin costo por request (frente a ~$4.2/mes de la API bajo el supuesto anterior) y con mejor privacidad del texto. La inversión es el entrenamiento en Mac (~55 min MPS) + hosting del Space en Hugging Face.


In [ ]:
import platform, torch, psutil
print(platform.platform(), "| RAM GB:", round(psutil.virtual_memory().total/1e9,1))
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no")

macOS-26.6.2-arm64-arm-64bit | RAM GB: 34.4
GPU: no


# 6. Aplicación web
La app es el gesto concreto del pitch: **alguien publica, alguien más podría leerlo; el sistema interviene con cuidado.**

1. **Laboratorio (Gradio / Hugging Face):** clasificador **A3** con umbral y probabilidad visibles. Para pruebas técnicas y la parte de métricas de la presentación. Límites: español, textos cortos (128 tokens / 500 caracteres), no sustituye moderación humana, puede fallar fuera de dominio.
2. **Consistencia cuaderno ↔ modelo:** smoke — limpio ≈ **0.03** / agresivo ≈ **0.995**. Ejemplos fijos en el laboratorio (limpio, disenso, agresivo, expulsión, contra-discurso, machismo/LGBTI-fobia).
3. **Blog LA PLAZA (Streamlit):** demo de lector (medio de noticias). El comentario se publica o **no se publica** según el modelo A3 en proceso + reglas de seguridad. Reemplaza al Gradio de blog cuando el cupo gratis de Spaces Gradio no alcanza.
4. **Blog estático (HTML):** misma estética en Hugging Face Static; filtro híbrido con reglas JS y, si responde, API del laboratorio. **Puede fallar** la parte del modelo si el Gradio está dormido o sin cupo: entonces solo aplican las reglas.
5. **Filtro híbrido:** (a) A3 (`HATEFUL`, umbral DEV ≈ 0,28); (b) reglas del sitio (insultos, amenazas, machismo/LGBTI-fobia sin grosería). No sustituyen a un moderador humano.
6. **Extensión de navegador:** desarrollo del equipo.

**URLs desplegadas**

- **Laboratorio (Gradio):** https://huggingface.co/spaces/tatisvivass/menos-odio-a-la-vista
- **LA PLAZA (Streamlit, demo principal del blog):** https://proyecto-2-eybyvirfnmkjnzroq5qy43.streamlit.app
- **LA PLAZA estático (respaldo visual):** https://huggingface.co/spaces/tatisvivass/la-plaza-demo · https://tatisvivass-la-plaza-demo.static.hf.space
- **Pesos A3:** https://huggingface.co/tatisvivass/menos-odio-a3-robertuito


In [ ]:
# Mini-demo Gradio local (opcional). En el Space se usa el mismo esquema con los pesos de A3.
# Requiere haber entrenado/cargado A3 (carpeta a3_model/ o artifacts/a3_model/).
import gradio as gr
from pathlib import Path
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
from pysentimiento.preprocessing import preprocess_tweet

_model_dir = next(p for p in [Path('a3_model'), Path('artifacts/a3_model')] if (p/'config.json').exists())
_tok = AutoTokenizer.from_pretrained(_model_dir)
_model = AutoModelForSequenceClassification.from_pretrained(_model_dir)
_device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
_model.to(_device).eval()
_THR = float(thr3) if 'thr3' in dir() else 0.28

def moderar(texto, umbral=_THR):
    texto = (texto or '').strip()
    if not texto:
        return 'Vacío', 0.0, 'Escribí un comentario.'
    enc = _tok(preprocess_tweet(texto, lang='es'), return_tensors='pt', truncation=True, max_length=128)
    enc = {k: v.to(_device) for k, v in enc.items()}
    with torch.no_grad():
        prob = torch.softmax(_model(**enc).logits, dim=-1)[0, 1].item()
    if prob >= umbral:
        return ('Con odio (se ocultaría)', round(prob, 4),
                'Este comentario se ocultaría por las normas de la comunidad. Opción: mostrar de todos modos / revisar.')
    return ('Sin odio (se mostraría)', round(prob, 4), 'Por debajo del umbral. El modelo no sustituye una decisión humana.')

# Descomentar para abrir la UI local:
# gr.Interface(moderar, [gr.Textbox(lines=3), gr.Slider(0.05, 0.95, value=_THR)],
#              [gr.Textbox(), gr.Number(), gr.Textbox()],
#              title='Menos odio a la vista').launch()
print('Función moderar() lista. Umbral:', _THR)
print(moderar('Qué buena nota, gracias.'))
print(moderar('Ojalá desaparezcan todos esos negros de mierda.'))


Función moderar() lista. Umbral: 0.2797354459762573
('Sin odio (se mostraría)', 0.0269, 'Por debajo del umbral. El modelo no sustituye una decisión humana.')
('Con odio (se ocultaría)', 0.9954, 'Este comentario se ocultaría por las normas de la comunidad. Opción: mostrar de todos modos / revisar.')



Loading weights: 100%|##########| 201/201 [00:00<00:00, 3534.13it/s]


# 7. Limitaciones, condiciones de uso y mejoras futuras
- **No es terapia ni prevención clínica:** solo reduce exposición a odio.
- **No es censura absoluta:** umbral + aviso + opción de mostrar / revisar.
- **Dominio:** español rioplatense / pandemia; en ALIA (YouTube/TikTok) F1≈0,28 y recall≈0,17.
- **Definiciones:** odio (piuba) ≠ multietiqueta de la línea base L.
- **Contra-discurso y negaciones:** falsos positivos/negativos frecuentes (HateCheck).
- **Privacidad:** el texto sale del navegador hacia la API del Space.
- **Contenido sensible** en el corpus y en las demos: cuidado al presentar.
- **Posible solapamiento** de preentrenamiento RoBERTuito con tuits UBA: declararlo.
- **Mejoras futuras:** más dialectos/plataformas; umbral por tipo de comunidad; comentario+título (H3); modelo on-device en la extensión; reentrenar con comentarios de blogs además de tuits.


# 8. Equipo
**Integrantes:** Esteban Salazar, Samuel Pico, David Vargas y Tatiana Vivas.

El desarrollo técnico de este entregable (cuaderno, modelado, evaluación y demos de app/blog) fue liderado y ejecutado de punta a punta en el marco del proyecto grupal.


# 9. Registro de revisión con IA
Cada observación de la IA se verificó con experimentos del equipo antes de presentarla como hallazgo. La IA solo recibió texto (no tono ni contexto audiovisual).

| Etapa | Prompt (resumen) | Evidencia | Crítica (síntesis) | Comprobación | Decisión |
|---|---|---|---|---|---|
| Fase 0 | Cuidador: ¿por qué no confiar en ocultar odio con aviso? | Pitch + HATEFUL + corpus | Censura injusta; no medir salud mental; sarcasmo; privacidad | HateCheck FP en contra-discurso; límites declarados | Filtro previo + humano; no prometer “salud mental” |
| EDA | ¿Qué implica la fuga por artículo? | test∩train=0 %; dev∩train=100 %; F1 regla 0,20 | DEV oficial no mide noticias nuevas | Celda 3.3 | Evaluación = `split_articulo_*` |
| Preparación | ¿La limpieza borra señales? | 4.2–4.5 | Conservar negaciones; no limpiar dos veces en A3 | Stopwords con negaciones; tokenizador uncased | A1 sin stopwords; A3 solo `preprocess_tweet` |
| Modelado | ¿Umbral por recall o precisión? | Tabla A1–A3–L; 0,5 vs 0,28 | Umbral bajo ↑FP en contra-discurso | F1 test 0,633→0,653 | A3 + umbral DEV; documentar trade-off |
| App | Cómo comunicar límites sin “seguridad total” | Blog + Gradio | Probar vacío, otro idioma, contra-discurso | Smoke 0,03 / 0,995 | Avisos claros + “mostrar de todos modos” |


# 10. Conclusiones del proyecto
1. **Problema:** filtrar comentarios con odio en español para reducir exposición, con transparencia y humano en el loop.
2. **Datos:** corpus `piuba-bigdata/contextualized_hate_speech`; evaluación principal por **artículo** (seed 42).
3. **Baseline léxica insuficiente:** F1≈0,20.
4. **Mejor modelo: A3 RoBERTuito** — F1 test **0,653**, AUC **0,898**, umbral DEV **0,28**.
5. **Comparación justa:** A3 > A2 > A1 > L (L con otra definición de etiqueta).
6. **Errores conocidos:** ortografía, odio implícito, negaciones, contra-discurso; peor fuera de dominio (ALIA).
7. **Producto:** laboratorio Gradio + blog **LA PLAZA** en Streamlit (demo principal) + estático HF como respaldo; URLs en §6. Extensión a cargo del equipo.
8. **Costo:** A3 local ~0,4 h/mes a 100k comentarios, sin $/request; alternativa OpenAI `gpt-4o-mini` ≈ **$4.2/mes** (tarifa 2026-11-06). Inversión: ~55 min fine-tuning en Apple Silicon (MPS).
9. **Ética de uso:** no sustituye moderación humana; declarar límites al usuario final.
